In [22]:
import pandas as pd

In [23]:
config = []
with open('Neo4j.config.txt', 'r') as configFile:
    for line in configFile.readlines():
        line = line.replace('\n', '')
        if line.startswith('NEO4J'):
            config.append(line.split('=')[1])

[url, username, password] = config

In [24]:
from py2neo import Graph, Node, Relationship, Subgraph
graph = Graph(url, auth=(username, password))

In [26]:
nurseData = pd.read_csv('../dataMyself/护工信息_已脱敏.csv', encoding='GBK')
patientData = pd.read_csv('../dataMyself/患者信息.csv')
orderData = pd.read_csv('../dataMyself/订单信息.csv', encoding='GBK')

In [27]:
# 创建医院节点
hospitalMap = {}
hospitalNodes = []
for ID, name in zip(orderData['医院ID'].unique(), orderData['医院名称'].unique()):
    node = Node('Hospital', 医院ID=str(ID), 医院名称=name)

    hospitalMap[ID] = node
    hospitalNodes.append(node)

graph.create(Subgraph(hospitalNodes))

In [28]:
# 创建服务商节点
providerMap = {}
providerNodes = []
for ID, name in zip(orderData['服务商ID'].unique(), orderData['服务商名称'].unique()):
    node = Node('Provider', 服务商ID=str(ID), 服务商名称=name)

    providerMap[ID] = node
    providerNodes.append(node)

graph.create(Subgraph(providerNodes))

In [29]:
# 服务商之间的竞争关系
relationships = []
providers = graph.nodes.match('Provider').all()
for i, left in enumerate(providers):
    for right in (providers[0:i] + providers[i + 1:]):
        relationships.append(Relationship(left, '竞争', right))

graph.create(Subgraph(relationships=relationships))

In [30]:
# 医院与服务商之间的合作关系
relationships = []
hospitals = graph.nodes.match('Hospital').all()
for hospital in hospitals:
    if hospital['医院ID'] == '2':
        provider = providerMap[1]
        relationships.append(Relationship(hospital, '合作', provider))
        relationships.append(Relationship(provider, '合作', hospital))
        provider = providerMap[15]
        relationships.append(Relationship(hospital, '合作', provider))
        relationships.append(Relationship(provider, '合作', hospital))
    elif hospital['医院ID'] == '3':
        provider = providerMap[21]
        relationships.append(Relationship(hospital, '合作', provider))
        relationships.append(Relationship(provider, '合作', hospital))
    else:
        provider = providerMap[21]
        relationships.append(Relationship(hospital, '合作', provider))
        relationships.append(Relationship(provider, '合作', hospital))
        provider = providerMap[45]
        relationships.append(Relationship(hospital, '合作', provider))
        relationships.append(Relationship(provider, '合作', hospital))

graph.create(Subgraph(relationships=relationships))

In [31]:
providerDecoder = {
    'NBAX': 1,
    'JSMSJ': 21,
    'XBJH': 15,
    'SHZF': 45
}
hospitalDecoder = {
    'FQ': 16,
    'YH': 3,
    'WT': 2
}

In [32]:
# 创建护工节点
nurseMap = {}
nurseNodes = []
relationships = []
for _, row in nurseData.iterrows():
    node = Node('Nurse')
    for col in nurseData.columns[0:-4]:
        node[col] = str(row[col])

    nurseMap[row['护工ID']] = node
    nurseNodes.append(node)

    hospitalNode = hospitalMap[hospitalDecoder[row['常驻医院']]]
    relationships.append(Relationship(node, '常驻', hospitalNode))

    providerNode = providerMap[providerDecoder[row['所属服务商']]]
    relationships.append(Relationship(node, '受雇', providerNode))
    relationships.append(Relationship(providerNode, '雇佣', node))

graph.create(Subgraph(nurseNodes, relationships))

In [33]:
# 创建患者节点
patientMap = {}
patientNodes = []
for _, row in patientData.iterrows():
    node = Node('Patient')
    patientID = str(row['患者ID']).zfill(4)
    node['患者ID'] = patientID
    for col in patientData.columns[1:-2]:
        node[col] = str(row[col])

    patientMap[patientID] = node
    patientNodes.append(node)

graph.create(Subgraph(patientNodes))

In [34]:
# 创建患者与护工之间的关系
relationships = []
with open('../dataMyself/护工_患者.txt', 'r') as file:
    for line in file.readlines():
        line = line.replace('\n', '')
        [nurseID, patientList] = line.split('->')
        patientList = eval(patientList)

        nurseNode = nurseMap[int(nurseID)]
        for patient in patientList:
            patientNode = patientMap[str(patient).zfill(4)]
            relationships.append(Relationship(nurseNode, '服务', patientNode))
            relationships.append(Relationship(patientNode, '被服务', nurseNode))

graph.create(Subgraph(relationships=relationships))

In [35]:
# 创建病区节点
areaNodes = []
areaMap = {}
for areaID, areaName in zip(list(orderData['病区ID'].unique()), list(orderData['病区名称'].unique())):
    node = Node('Area')
    node['病区ID'] = str(areaID)
    node['病区名称'] = areaName

    tmp = areaName.split(' ')
    buildingNumber, floor = tmp[0], tmp[1]
    node['楼号'] = buildingNumber[0]
    node['楼层'] = floor[0]

    areaNodes.append(node)
    areaMap[areaID] = node

graph.create(Subgraph(areaNodes))

In [36]:
# 创建病区与医院的关系
relationships = []
for hospitalNode in graph.nodes.match('Hospital').all():
    for areaID in orderData[orderData['医院ID'] == int(hospitalNode['医院ID'])]['病区ID'].unique():
        areaNode = areaMap[areaID]
        relationships.append(Relationship(areaNode, '位于', hospitalNode))

graph.create(Subgraph(relationships=relationships))

In [37]:
# 创建护工与病区的关系
relationships = []
for areaNode in graph.nodes.match('Area').all():
    for nurseID in orderData[orderData['病区ID'] == int(areaNode['病区ID'])]['护工ID'].unique():
        nurseNode = nurseMap[nurseID]
        relationships.append(Relationship(nurseNode, '负责', areaNode))

graph.create(Subgraph(relationships=relationships))

In [38]:
def transformOrderID(orderID, createTime):
    [date, time] = createTime.split(' ')
    [year, month, day] = date.split('/')
    [hour, minute] = time.split(':')
    month = '0' + month if len(month) == 1 else month
    day = '0' + day if len(day) == 1 else day
    hour = '0' + hour if len(hour) == 1 else hour

    return "{:.0f}".format(orderID)[0:5] + year + month + day + hour + minute

In [39]:
# 订单属性
orderProperty = ['订单号', '订单类型', '服务项目', '科室', '床位号', '创建时间', '接单时间', '服务开始时间',
                 '服务结束时间', '关闭时间', '服务天数', '费用', '已支付费用', '已退款费用']

orderNodes = []
orderMap = {}
relationships = []
for i, row in orderData.iterrows():
    node = Node('Order')
    for col in orderProperty:
        if col == '订单号':
            node[col] = transformOrderID(row['订单号'], row['创建时间'])
        else:
            node[col] = str(row[col])
    orderNodes.append(node)
    orderMap[i] = node

    hospitalNode = hospitalMap[row['医院ID']]
    relationships.append(Relationship(node, '位于', hospitalNode))

    areaNode = areaMap[row['病区ID']]
    relationships.append(Relationship(node, '位于', areaNode))

    nurseNode = nurseMap[row['护工ID']]
    relationships.append(Relationship(nurseNode, '接取', node))

graph.create(Subgraph(orderNodes, relationships))

In [40]:
# 创建患者与订单、病区、医院的关系
relationships = []
with open('../dataMyself/患者_订单.txt', 'r') as file:
    for line in file.readlines():
        [patientID, orderList] = line.replace('\n', '').split('->')

        patientID = patientID.zfill(4)
        patientNode = patientMap[patientID]

        for orderIndex in eval(orderList):
            order = orderData.iloc[orderIndex]

            orderNode = orderMap[orderIndex]
            relationships.append(Relationship(patientNode, '创建', orderNode))

            hospitalNode = hospitalMap[order['医院ID']]
            relationships.append(Relationship(patientNode, '位于', hospitalNode))

            areaNode = areaMap[order['病区ID']]
            relationships.append(Relationship(patientNode, '位于', areaNode))

graph.create(Subgraph(relationships=relationships))